In [0]:

# The path structure is always /Volumes/Catalog/Schema/VolumeName/FileName
file_path ="/Volumes/workspace/default/task2"

df_health = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# Verify the data
display(df_health.limit(50))

age,sex,bmi,children,smoker,region,charges
19,female,27.9,0,yes,southwest,16884.924
18,male,33.77,1,no,southeast,1725.5523
28,male,33.0,3,no,southeast,4449.462
33,male,22.705,0,no,northwest,21984.47061
32,male,28.88,0,no,northwest,3866.8552
31,female,25.74,0,no,southeast,3756.6216
46,female,33.44,1,no,southeast,8240.5896
37,female,27.74,3,no,northwest,7281.5056
37,male,29.83,2,no,northeast,6406.4107
60,female,25.84,0,no,northwest,28923.13692


In [0]:
from pyspark.ml.feature import StringIndexer

# We tell Spark which columns are text and what the new number columns should be named
indexer = StringIndexer(inputCols=["sex", "smoker", "region"], 
                        outputCols=["sex_idx", "smoker_idx", "region_idx"])

# Apply the transformation
df_numeric = indexer.fit(df_health).transform(df_health)

# Look at the new columns at the end of the table
display(df_numeric.limit(5))

age,sex,bmi,children,smoker,region,charges,sex_idx,smoker_idx,region_idx
19,female,27.9,0,yes,southwest,16884.924,1.0,1.0,2.0
18,male,33.77,1,no,southeast,1725.5523,0.0,0.0,0.0
28,male,33.0,3,no,southeast,4449.462,0.0,0.0,0.0
33,male,22.705,0,no,northwest,21984.47061,0.0,0.0,1.0
32,male,28.88,0,no,northwest,3866.8552,0.0,0.0,1.0


In [0]:
from pyspark.ml.feature import VectorAssembler

# Pick the columns that affect insurance cost
input_cols = ["age", "bmi", "children", "sex_idx", "smoker_idx", "region_idx"]

# Group them together
assembler = VectorAssembler(inputCols=input_cols, outputCol="features")

# Create the final data for the Machine Learning model
final_data = assembler.transform(df_numeric).select("features", "charges")

display(final_data.limit(5))

features,charges
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""19.0"",""27.9"",""0.0"",""1.0"",""1.0"",""2.0""]}",16884.924
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""33.77"",""1.0"",""0.0"",""0.0"",""0.0""]}",1725.5523
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""28.0"",""33.0"",""3.0"",""0.0"",""0.0"",""0.0""]}",4449.462
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""33.0"",""22.705"",""0.0"",""0.0"",""0.0"",""1.0""]}",21984.47061
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""32.0"",""28.88"",""0.0"",""0.0"",""0.0"",""1.0""]}",3866.8552


In [0]:
from pyspark.ml.regression import LinearRegression

# Split Data (80% Train, 20% Test)
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

# Train Model
lr = LinearRegression(featuresCol="features", labelCol="charges")
lr_model = lr.fit(train_data)

# Evaluate
results = lr_model.evaluate(test_data)
print(f"R2 Score: {results.r2:.4f}")
print(f"Mean Absolute Error: ${results.meanAbsoluteError:.2f}")

R2 Score: 0.7192
Mean Absolute Error: $4477.69


In [0]:
predictions = lr_model.transform(test_data)
display(predictions.select("charges", "prediction"))

charges,prediction
1135.9407,2764.5044737724165
1141.4451,4110.233812367082
1149.3959,6054.065079226042
1532.4697,4272.3637647988
1837.2819,6947.304508554211
2322.6218,5502.035528775639
3704.3545,5518.985502789972
5699.8375,8667.711633474893
27346.04207,10961.011745928583
9504.3103,14400.097833448282


Databricks visualization. Run in Databricks to view.

In [0]:
# Split the data: 80% for training the model, 20% for testing its accuracy
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_data.count()}")
print(f"Testing rows: {test_data.count()}")

Training rows: 1108
Testing rows: 230


In [0]:
# This will force the schema to appear and show you the actual data
display(train_data.limit(5))

# This will print the column structure
train_data.printSchema()

features,charges
"{""type"":""0"",""size"":""6"",""indices"":[""0"",""1""],""values"":[""18.0"",""23.21""]}",1121.8739
"{""type"":""0"",""size"":""6"",""indices"":[""0"",""1""],""values"":[""18.0"",""30.14""]}",1131.5066
"{""type"":""0"",""size"":""6"",""indices"":[""0"",""1""],""values"":[""18.0"",""33.66""]}",1136.3994
"{""type"":""0"",""size"":""6"",""indices"":[""0"",""1""],""values"":[""18.0"",""34.1""]}",1137.011
"{""type"":""0"",""size"":""6"",""indices"":[""0"",""1""],""values"":[""18.0"",""34.43""]}",1137.4697


root
 |-- features: vectorudt (nullable = true)
 |-- charges: double (nullable = true)



In [0]:
from pyspark.ml.regression import LinearRegression

# 1. Initialize the algorithm
# We tell it 'features' are the inputs and 'charges' is what we want to predict
lr = LinearRegression(featuresCol="features", labelCol="charges")

# 2. Train the model using the training data
lr_model = lr.fit(train_data)

print("Model Training Complete!")

Model Training Complete!


In [0]:
# 1. Make predictions on the test data
predictions = lr_model.transform(test_data)

# 2. Evaluate the performance
training_summary = lr_model.summary

print(f"RMSE (Average Error): ${training_summary.rootMeanSquaredError:.2f}")
print(f"R2 (Accuracy Score): {training_summary.r2:.4f}")

# 3. Compare Actual vs. Predicted
display(predictions.select("charges", "prediction").limit(10))

RMSE (Average Error): $5978.80
R2 (Accuracy Score): 0.7563


charges,prediction
1135.9407,2764.5044737724165
1141.4451,4110.233812367082
1149.3959,6054.065079226042
1532.4697,4272.3637647988
1837.2819,6947.304508554211
2322.6218,5502.035528775639
3704.3545,5518.985502789972
5699.8375,8667.711633474893
27346.04207,10961.011745928583
9504.3103,14400.097833448282
